<div style="background: linear-gradient(135deg, #e63946 0%, #f1faee 100%); padding: 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: white; margin: 0; font-size: 2em;">⚖️ 01 — As Punições Injustas da Acurácia</h1>
  <p style="color: #d1f2eb; margin: 8px 0 0 0; font-size: 1.1em;">Fase 08 · Bloco 02 · Funções de Perda</p>
</div>

## 🎯 Objetivos

- Entender por que a CrossEntropy pura falha em imagens médicas
- Implementar a Dice Loss e a Focal Loss

---

## 1️⃣ O Problema do Desbalanceamento de Classes

Imagine segmentar um tumor minúsculo. 99% da imagem é 'fundo' (preto) e 1% é 'tumor' (branco). 
Se a rede chutar que a imagem toda é fundo, a Cross-Entropy vai dar 99% de acurácia. A rede vira preguiçosa.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Dice Loss: Ignora os 'True Negatives' (O fundo gigante)
def dice_loss(pred, target, smooth=1e-6):
    # Pred deve vir da camada sem ativação (logits), então passamos Sigmoid
    pred = torch.sigmoid(pred)
    
    # Achatamos os tensores
    pred = pred.view(-1)
    target = target.view(-1)
    
    intersecao = (pred * target).sum()
    dice_score = (2. * intersecao + smooth) / (pred.sum() + target.sum() + smooth)
    
    return 1 - dice_score

print('Dice Loss criada! Muito usada na área médica.')

## 2️⃣ Focal Loss

Outra forma de consertar o desbalanceamento. A Focal Loss pune muito mais a rede quando ela erra algo com 'alta confiança'. 

In [ ]:
def focal_loss(pred, target, alpha=0.8, gamma=2):
    bce = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
    pt = torch.exp(-bce)  # Probabilidade da classe correta
    focal = alpha * (1 - pt)**gamma * bce
    return focal.mean()

print('Focal Loss criada! Muito usada em instâncias difíceis (bordas de objetos).')

## 🏋️ Questões

### Q1. Em vários artigos, pesquisadores usam a soma: `Loss = 0.5 * BCE + 0.5 * Dice`. Por que combinar as duas?

### Q2. Se a rede prever uma máscara perfeita igual ao Ground Truth, qual será o valor do `dice_score`? E o `dice_loss`?